In [ ]:
!pip install google-play-scraper

In [ ]:
from google_play_scraper import reviews, Sort
import pandas as pd

result, _ = reviews(
    'com.moniepoint.personal',
    lang='en',
    country='ng',
    sort=Sort.NEWEST,
    count=25000
)

df = pd.DataFrame(result)
print(df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
df = df[['content', 'score', 'at']].dropna()
df = df[df['content'].str.strip() != '']
df['at'] = pd.to_datetime(df['at'])
print(df.shape)
df.head()

In [ ]:
!pip install vaderSentiment

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

df['sentiment_score'] = df['content'].apply(
    lambda x: analyzer.polarity_scores(x)['compound']
)

df['sentiment_label'] = df['sentiment_score'].apply(
    lambda x: 'Positive' if x >= 0.05 else ('Negative' if x <= -0.05 else 'Neutral')
)

print(df['sentiment_label'].value_counts())

In [ ]:
print(df['sentiment_score'].describe())
print(df['score'].describe())

In [ ]:
import matplotlib.pyplot as plt

df['sentiment_label'].value_counts().plot(
    kind='bar',
    color=['green', 'red', 'grey'],
    title='Moniepoint Sentiment Distribution'
)
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')
plt.tight_layout()
plt.savefig('moniepoint_sentiment_distribution.png', dpi=300)
plt.show()

In [ ]:
df['month'] = df['at'].dt.to_period('M')
monthly = df.groupby('month')['score'].mean()

monthly.plot(
    kind='line',
    title='Moniepoint Average Rating Over Time',
    color='blue',
    figsize=(12,5)
)
plt.ylabel('Average Rating')
plt.xlabel('Month')
plt.tight_layout()
plt.savefig('moniepoint_rating_trend.png', dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Define the desired date range for filtering using Period objects
start_period = pd.Period('2025-08', freq='M')
end_period = pd.Period('2026-04', freq='M')

# Filter the DataFrame based on the 'month' column for the specified range
filtered_df = df[(df['month'] >= start_period) & (df['month'] <= end_period)].copy()

# Group by the 'month' column of the filtered DataFrame and count reviews
# The 'month' column is already of Period type, ensuring correct chronological grouping.
review_volume = filtered_df.groupby('month')['content'].count()

# Convert the PeriodIndex to a string format for better display on the x-axis
# e.g., '2025-08' becomes 'Aug 2025'
review_volume.index = review_volume.index.strftime('%b %Y')

# Plotting the review volume
review_volume.plot(
    kind='bar',
    title='Moniepoint Review Volume Over Time (Aug 2025 - Apr 2026)',
    color='blue',
    figsize=(12,5)
)

plt.ylabel('Number of Reviews')
plt.xlabel('Month')
plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for better readability
plt.tight_layout()
plt.savefig('moniepoint_review_volume.png', dpi=300)
plt.show()

In [ ]:
from scipy import stats

corr, pvalue = stats.pearsonr(df['score'], df['sentiment_score'])
print(f'Pearson Correlation: {corr:.3f}')
print(f'P-Value: {pvalue:.5f}')

In [ ]:
!pip install wordcloud

from wordcloud import WordCloud

positive_text = ' '.join(df[df['sentiment_label']=='Positive']['content'])
negative_text = ' '.join(df[df['sentiment_label']=='Negative']['content'])

WordCloud(width=800, height=400, background_color='white').generate(positive_text).to_file('positive_wordcloud.png')
WordCloud(width=800, height=400, background_color='black').generate(negative_text).to_file('negative_wordcloud.png')

print("Word clouds saved!")

In [ ]:
opay_result, _ = reviews(
    'team.opay.pay',
    lang='en',
    country='ng',
    sort=Sort.NEWEST,
    count=25000
)

palmpay_result, _ = reviews(
    'com.transsnet.palmpay',
    lang='en',
    country='ng',
    sort=Sort.NEWEST,
    count=25000
)

opay_df = pd.DataFrame(opay_result)[['content','score','at']].dropna()
palmpay_df = pd.DataFrame(palmpay_result)[['content','score','at']].dropna()

for frame in [opay_df, palmpay_df]:
    frame['sentiment_score'] = frame['content'].apply(
        lambda x: analyzer.polarity_scores(x)['compound']
    )
    frame['sentiment_label'] = frame['sentiment_score'].apply(
        lambda x: 'Positive' if x >= 0.05 else ('Negative' if x <= -0.05 else 'Neutral')
    )

print("Opay shape:", opay_df.shape)
print("Palmpay shape:", palmpay_df.shape)
print("\nOpay sentiment:")
print(opay_df['sentiment_label'].value_counts())
print("\nPalmpay sentiment:")
print(palmpay_df['sentiment_label'].value_counts())

In [ ]:
from scipy.stats import mannwhitneyu

u1, p1 = mannwhitneyu(df['sentiment_score'], opay_df['sentiment_score'])
u2, p2 = mannwhitneyu(df['sentiment_score'], palmpay_df['sentiment_score'])

print(f'Moniepoint vs Opay — U: {u1:.0f}, P: {p1:.5f}, Significant: {p1 < 0.05}')
print(f'Moniepoint vs Palmpay — U: {u2:.0f}, P: {p2:.5f}, Significant: {p2 < 0.05}')

In [ ]:
print("Moniepoint sentiment:")
print(df['sentiment_label'].value_counts())
print("\nMoniepoint shape:", df.shape)

In [ ]:
print("Average Ratings:")
print(f"Moniepoint: {df['score'].mean():.2f}")
print(f"Opay: {opay_df['score'].mean():.2f}")
print(f"Palmpay: {palmpay_df['score'].mean():.2f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Nigerian Fintech Sentiment Dashboard', fontsize=16, fontweight='bold', y=1.02)

platforms = ['Moniepoint', 'Opay', 'Palmpay']
x = np.arange(len(platforms))
width = 0.25

axes[0].bar(x - width, [82.5, 82.7, 77.9], width, label='Positive', color='#1D9E75', zorder=3)
axes[0].bar(x, [11.8, 13.3, 12.1], width, label='Neutral', color='#888780', zorder=3)
axes[0].bar(x + width, [5.7, 4.1, 10.0], width, label='Negative', color='#E24B4A', zorder=3)
axes[0].set_title('Sentiment Distribution', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(platforms)
axes[0].set_ylabel('Percentage (%)')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3, zorder=0)

axes[1].bar(platforms, [5.7, 4.1, 10.0], color=['#378ADD','#1D9E75','#E24B4A'], zorder=3)
axes[1].set_title('Negative Sentiment %', fontweight='bold')
axes[1].set_ylabel('Percentage (%)')
axes[1].grid(axis='y', alpha=0.3, zorder=0)

axes[2].bar(platforms, [4.44, 4.61, 4.21], color=['#378ADD','#1D9E75','#E24B4A'], zorder=3)
axes[2].set_title('Average Rating', fontweight='bold')
axes[2].set_ylabel('Rating (out of 5)')
axes[2].set_ylim(3.8, 5.0)
axes[2].grid(axis='y', alpha=0.3, zorder=0)

plt.tight_layout()
plt.savefig('sentiment_dashboard.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
switching_keywords = ['switched', 'left', 'moved from', 'used to use', 'used to be', 'changed from', 'came from']

switching = opay_df[opay_df['content'].str.contains('|'.join(switching_keywords), case=False, na=False)]
print(f"Switching mentions found: {len(switching)}")
print(switching['content'].head(10))

In [ ]:
df.to_csv('moniepoint_reviews.csv', index=False)
opay_df.to_csv('opay_reviews.csv', index=False)
palmpay_df.to_csv('palmpay_reviews.csv', index=False)
print("All files saved!")

In [ ]:
!pip install wordcloud

from wordcloud import WordCloud
import matplotlib.pyplot as plt

positive_text = ' '.join(df[df['sentiment_label']=='Positive']['content'])
negative_text = ' '.join(df[df['sentiment_label']=='Negative']['content'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(WordCloud(width=800, height=400, background_color='white', colormap='Greens').generate(positive_text))
axes[0].set_title('What Users Love', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(WordCloud(width=800, height=400, background_color='white', colormap='Reds').generate(negative_text))
axes[1].set_title('What Users Hate', fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('wordclouds.png', dpi=300)
plt.show()

In [ ]:
print("Moniepoint:")
print(df['sentiment_score'].describe().round(3))
print("\nOpay:")
print(opay_df['sentiment_score'].describe().round(3))
print("\nPalmpay:")
print(palmpay_df['sentiment_score'].describe().round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, frame, name, color in zip(
    axes,
    [df, opay_df, palmpay_df],
    ['Moniepoint', 'Opay', 'Palmpay'],
    ['#378ADD', '#1D9E75', '#E24B4A']
):
    ax.hist(frame['sentiment_score'], bins=50, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(f'{name} Sentiment Distribution', fontweight='bold')
    ax.set_xlabel('Sentiment Score')
    ax.set_ylabel('Number of Reviews')
    ax.axvline(x=0.05, color='green', linestyle='--', alpha=0.5, label='Positive threshold')
    ax.axvline(x=-0.05, color='red', linestyle='--', alpha=0.5, label='Negative threshold')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('sentiment_distribution_histogram.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
from scipy import stats

corr_m, p_m = stats.pearsonr(df['score'], df['sentiment_score'])
corr_o, p_o = stats.pearsonr(opay_df['score'], opay_df['sentiment_score'])
corr_p, p_p = stats.pearsonr(palmpay_df['score'], palmpay_df['sentiment_score'])

print(f"Moniepoint — r: {corr_m:.3f}, p: {p_m:.5f}")
print(f"Opay — r: {corr_o:.3f}, p: {p_o:.5f}")
print(f"Palmpay — r: {corr_p:.3f}, p: {p_p:.5f}")

In [ ]:
oct_reviews = df[
    (df['at'].dt.month == 10) &
    (df['at'].dt.year == 2025)
]

print(f"October review count: {len(oct_reviews)}")
print(f"October average rating: {oct_reviews['score'].mean():.2f}")
print(f"October sentiment breakdown:")
print(oct_reviews['sentiment_label'].value_counts())

print("\nSample negative October reviews:")
print(oct_reviews[oct_reviews['sentiment_label']=='Negative']['content'].sample(20).tolist())

In [ ]:
df.groupby(df['at'].dt.to_period('M'))['content'].count().plot(
    kind='bar', figsize=(14,5), color='#378ADD',
    title='Monthly Review Volume — Moniepoint'
)
plt.savefig('review_volume.png', dpi=300)
plt.show()

In [ ]:
from google_play_scraper import reviews, Sort
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

result, _ = reviews(
    'com.moniepoint.personal',
    lang='en',
    country='ng',
    sort=Sort.NEWEST,
    count=5000
)

df = pd.DataFrame(result)[['content','score','at']].dropna()
df['at'] = pd.to_datetime(df['at'])

df['sentiment_score'] = df['content'].apply(
    lambda x: analyzer.polarity_scores(x)['compound']
)
df['sentiment_label'] = df['sentiment_score'].apply(
    lambda x: 'Positive' if x >= 0.05 else ('Negative' if x <= -0.05 else 'Neutral')
)

positive_reviews = df[df['sentiment_label']=='Positive']['content']
negative_reviews = df[df['sentiment_label']=='Negative']['content']

print("Done! Shape:", df.shape)

In [ ]:
print(positive_reviews.sample(20).tolist())
print(negative_reviews.sample(20).tolist())

In [ ]:
opay_positive = opay_df[opay_df['sentiment_label']=='Positive']['content']
opay_negative = opay_df[opay_df['sentiment_label']=='Negative']['content']

print("OPAY POSITIVE SAMPLE:")
print(opay_positive.sample(20).tolist())

print("\nOPAY NEGATIVE SAMPLE:")
print(opay_negative.sample(20).tolist())

In [ ]:
switching_keywords = ['switched', 'left', 'moved from', 'used to use', 'used to be', 'changed from', 'came from', 'left moniepoint', 'left palmpay', 'switched from']

switching = opay_df[opay_df['content'].str.contains('|'.join(switching_keywords), case=False, na=False)]

print(f"Switching mentions found: {len(switching)}")
print(switching['content'].tolist())

In [ ]:
opay_df['at'] = pd.to_datetime(opay_df['at'])
opay_df['month'] = opay_df['at'].dt.to_period('M')

opay_df.groupby('month')['score'].mean().plot(
    kind='line',
    color='#1D9E75',
    linewidth=2.5,
    marker='o',
    markersize=4,
    figsize=(14,5),
    title='Opay Average Rating Over Time'
)
import matplotlib.pyplot as plt
plt.ylabel('Average Rating')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('opay_rating_trend.png', dpi=300)5
plt.show()

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(WordCloud(width=800, height=400, background_color='white', colormap='Greens').generate(' '.join(opay_positive)))
axes[0].set_title("What Opay Users Love", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(WordCloud(width=800, height=400, background_color='white', colormap='Reds').generate(' '.join(opay_negative)))
axes[1].set_title("What Opay Users Hate", fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('opay_wordclouds.png', dpi=300)
plt.show()

In [ ]:
palmpay_positive = palmpay_df[palmpay_df['sentiment_label']=='Positive']['content']
palmpay_negative = palmpay_df[palmpay_df['sentiment_label']=='Negative']['content']

print("PALMPAY POSITIVE SAMPLE:")
print(palmpay_positive.sample(20).tolist())

print("\nPALMPAY NEGATIVE SAMPLE:")
print(palmpay_negative.sample(20).tolist())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

platforms = ['Moniepoint', 'Opay', 'Palmpay']
ratings = [4.44, 4.61, 4.21]
colors = ['#378ADD', '#1D9E75', '#E24B4A']

axes[0].bar(platforms, ratings, color=colors, zorder=3)
axes[0].set_title('Average Rating Comparison', fontweight='bold')
axes[0].set_ylabel('Average Rating (out of 5)')
axes[0].set_ylim(3.8, 5.0)
axes[0].grid(axis='y', alpha=0.3, zorder=0)

categories = ['Positive', 'Neutral', 'Negative']
moniepoint_vals = [82.5, 11.8, 5.7]
opay_vals = [82.7, 13.3, 4.1]
palmpay_vals = [77.9, 12.1, 10.0]

x = np.arange(len(categories))
width = 0.25

axes[1].bar(x - width, moniepoint_vals, width, label='Moniepoint', color='#378ADD', zorder=3)
axes[1].bar(x, opay_vals, width, label='Opay', color='#1D9E75', zorder=3)
axes[1].bar(x + width, palmpay_vals, width, label='Palmpay', color='#E24B4A', zorder=3)
axes[1].set_title('Sentiment Distribution Comparison', fontweight='bold')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(categories)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3, zorder=0)

plt.tight_layout()
plt.savefig('three_way_comparison.png', dpi=300)
plt.show()